# Basic Manual Setup

> **Before we begin, make sure that you are running an observability suite (Grafana etc.).**  
> See the project root's `README.md` for more tips on that.

|||
|--|--|
|**Main Goal**| Experience the basic of OpenTelemetry tracing. |
|**Approach**| Here we setup an **explicit, manual, tracing configuration**. In practice, you probably can avoid this using auto-instrumentation. |



In [1]:
!cd .. && pip install --group "runtime"

## Manual OpenTelemetry tracing

> **Load local environment variables**

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

> **We configure logging to see OpenTelemetry external transport**

In [3]:
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)
logger = logging.getLogger("opentelemetry")
logger.setLevel(logging.DEBUG)

> **Here we set the tracer up with two providers: `console` and `otlp` (external collector)**

In [4]:
import os

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
otlp_endpoint = os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"]
provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter(endpoint=otlp_endpoint)))

trace.set_tracer_provider(provider)
tracer = trace.get_tracer("demo.example.service.notebook")

> **This is some basic "business" logic we want to trace**

In [5]:
from random import randint

def flip_coin() -> int:
    return randint(0, 1)

In [6]:
flip_coin()

0

> **Here we setup the span with an explicit context block: `with tracer.start_as_current_span(...) ...`**

In [7]:
def coin_batch(size: int) -> list[(int, int)]:
    with tracer.start_as_current_span(
        f"coin-flipping-batch-x-size-{size}"
    ) as span:
        outcomes = [(i, flip_coin()) for i in range(size)]
        span.set_attribute("operation.request.size", size)
        span.set_attribute("operation.result.outcomes", outcomes)
    return outcomes

> **Running the code therefore records the span and pushes it to the configured trace providers**

In [8]:
coin_batch(5)

{
    "name": "coin-flipping-batch-x-size-5",
    "context": {
        "trace_id": "0x37417055e1bde6e1b1185108ecdbcb9c",
        "span_id": "0xf1568ed06ec5a1a1",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-10-28T11:25:13.624423Z",
    "end_time": "2026-10-28T11:25:13.624474Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "operation.request.size": 5,
        "operation.result.outcomes": [
            [
                0,
                1
            ],
            [
                1,
                0
            ],
            [
                2,
                1
            ],
            [
                3,
                0
            ],
            [
                4,
                0
            ]
        ]
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.nam

[(0, 1), (1, 0), (2, 1), (3, 0), (4, 0)]

> **As a result, with an observability suite (Grafana and etc) running, you should find the traces in the platform!**  
> There may be a small delay before you receive the traces.